In [1]:
%useLatestDescriptors
%use dataframe
%use kandy

In [3]:
val dataPath = "https://raw.githubusercontent.com/Kotlin/dataframe/refs/heads/master/examples/idea-examples/movies/src/main/resources/movies.csv"
val rawMovies = DataFrame.read(dataPath)

rawMovies

movieId,title,genres
9b30aff7943f44579e92c261f3adc193,Women in Black (1997),Fantasy|Suspenseful|Comedy
2a1ba1fc5caf492a80188e032995843e,Bumblebee Movie (2007),Comedy|Jazz|Family|Animation
f44ceb4771504342bb856d76c112d5a6,Magical School Boy and the Rock of Wi...,Fantasy|Growing up|Magic
43d02fb064514ff3bd30d1e3a7398357,Master of the Jewlery: The Company of...,Fantasy|Magic|Suspenseful
6aa0d26a483148998c250b9c80ddf550,Sun Conflicts: Part IV: A Novel Espai...,Fantasy
eace16e59ce24eff90bf8924eb6a926c,The Outstanding Bulk (2008),Fantasy|Superhero|Family
ae916bc4844a4bb7b42b70d9573d05cd,In Automata (2014),Horror|Existential
c1f0a868aeb44c5ea8d154ec3ca295ac,Interplanetary (2014),Sci-fi|Futuristic
9595b771f87f42a3b8dd07d91e7cb328,Woods Run (1994),Family|Drama
aa9fc400e068443488b259ea0802a975,Anthropod-Dude (2002),Superhero|Fantasy|Family|Growing up


In [9]:
val moviesWithYear = rawMovies.add("year"){ "(\\d{4})".toRegex().findAll(title).lastOrNull()?.value?.toInt() ?: -1 }
moviesWithYear

movieId,title,genres,year
9b30aff7943f44579e92c261f3adc193,Women in Black (1997),Fantasy|Suspenseful|Comedy,1997
2a1ba1fc5caf492a80188e032995843e,Bumblebee Movie (2007),Comedy|Jazz|Family|Animation,2007
f44ceb4771504342bb856d76c112d5a6,Magical School Boy and the Rock of Wi...,Fantasy|Growing up|Magic,2001
43d02fb064514ff3bd30d1e3a7398357,Master of the Jewlery: The Company of...,Fantasy|Magic|Suspenseful,2001
6aa0d26a483148998c250b9c80ddf550,Sun Conflicts: Part IV: A Novel Espai...,Fantasy,1977
eace16e59ce24eff90bf8924eb6a926c,The Outstanding Bulk (2008),Fantasy|Superhero|Family,2008
ae916bc4844a4bb7b42b70d9573d05cd,In Automata (2014),Horror|Existential,2014
c1f0a868aeb44c5ea8d154ec3ca295ac,Interplanetary (2014),Sci-fi|Futuristic,2014
9595b771f87f42a3b8dd07d91e7cb328,Woods Run (1994),Family|Drama,1994
aa9fc400e068443488b259ea0802a975,Anthropod-Dude (2002),Superhero|Fantasy|Family|Growing up,2002


In [10]:
moviesWithYear.year.describe()

name,type,count,unique,nulls,top,freq,mean,std,min,median,max
year,Int,20,16,0,1997,2,1998.400000,16.040738,1950,2001,2022


In [12]:
val movies = moviesWithYear
    .update("title") {
        "\\s*\\(\\d{4}\\)\\s*$".toRegex().replace(title, "")
    }

movies

movieId,title,genres,year
9b30aff7943f44579e92c261f3adc193,Women in Black,Fantasy|Suspenseful|Comedy,1997
2a1ba1fc5caf492a80188e032995843e,Bumblebee Movie,Comedy|Jazz|Family|Animation,2007
f44ceb4771504342bb856d76c112d5a6,Magical School Boy and the Rock of Wi...,Fantasy|Growing up|Magic,2001
43d02fb064514ff3bd30d1e3a7398357,Master of the Jewlery: The Company of...,Fantasy|Magic|Suspenseful,2001
6aa0d26a483148998c250b9c80ddf550,Sun Conflicts: Part IV: A Novel Espair,Fantasy,1977
eace16e59ce24eff90bf8924eb6a926c,The Outstanding Bulk,Fantasy|Superhero|Family,2008
ae916bc4844a4bb7b42b70d9573d05cd,In Automata,Horror|Existential,2014
c1f0a868aeb44c5ea8d154ec3ca295ac,Interplanetary,Sci-fi|Futuristic,2014
9595b771f87f42a3b8dd07d91e7cb328,Woods Run,Family|Drama,1994
aa9fc400e068443488b259ea0802a975,Anthropod-Dude,Superhero|Fantasy|Family|Growing up,2002


In [19]:
movies
    .filter { year >= 1920 && genres != "(no genres listed)" }
    .split { genres }.by("|").intoRows()
    .groupBy { year and genres }.count()
    .sortBy { it["count"].desc() }

year,genres,count
2001,Fantasy,2
2001,Magic,2
1997,Fantasy,1
1997,Suspenseful,1
1997,Comedy,1
2007,Comedy,1
2007,Jazz,1
2007,Family,1
2007,Animation,1
2001,Growing up,1


In [22]:
movies
    .filter { year >= 1920 && genres != "(no genres listed)" }
    .split { genres }.by("|").intoRows()
    .sortBy { year and genres }
    .groupBy { year }.pivotCounts { genres }

year,genres,,,,,,,,,,,,,,,
,Fantasy,Horror,Drama,Family,Animation,Comedy,History,Romance,Suspenseful,Growing up,Magic,Superhero,Jazz,Existential,Futuristic,Sci-fi
1977,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1980,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1984,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1994,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0
1995,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0
1997,1,0,1,1,0,1,1,1,1,0,0,0,0,0,0,0
2001,2,0,0,0,0,0,0,0,1,1,2,0,0,0,0,0
2002,1,0,0,1,0,0,0,0,0,1,0,1,0,0,0,0
2003,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0


In [25]:
val dataPath = "https://raw.githubusercontent.com/Kotlin/dataframe/refs/heads/master/examples/idea-examples/movies/src/main/resources/tags.csv"
val tags = DataFrame.read(dataPath)
tags

userId,movieId,tag,timestamp
3,9595b771f87f42a3b8dd07d91e7cb328,classic,1439472355
3,6aa0d26a483148998c250b9c80ddf550,sci-fi,1439472256
4,f24327f2b05147b197ca34bf13ae3524,dark comedy,1573943598
4,ae916bc4844a4bb7b42b70d9573d05cd,great dialogue,1573943604
4,f24327f2b05147b197ca34bf13ae3524,so bad it's good,1573943455
4,d4a325ab648a42c4a2d6f35dfabb387f,tense,1573943077
4,ae916bc4844a4bb7b42b70d9573d05cd,artificial intelligence,1573942979
4,ae916bc4844a4bb7b42b70d9573d05cd,philosophical,1573943033
4,c1f0a868aeb44c5ea8d154ec3ca295ac,tense,1573943042
4,22d20c2ba11d44cab83aceea39dc00bd,so bad it's good,1573942965


In [26]:
tags.schema()

userId: Int
movieId: String
tag: String
timestamp: Int

In [27]:
val moviesWithTags = movies.leftJoin(tags) { movieId }
moviesWithTags

movieId,title,genres,year,userId,tag,timestamp
9b30aff7943f44579e92c261f3adc193,Women in Black,Fantasy|Suspenseful|Comedy,1997,19,Oscar (Best Supporting Actress),1446909853
2a1ba1fc5caf492a80188e032995843e,Bumblebee Movie,Comedy|Jazz|Family|Animation,2007,20,bah,1155082282
f44ceb4771504342bb856d76c112d5a6,Magical School Boy and the Rock of Wi...,Fantasy|Growing up|Magic,2001,19,fantasy,1445286144
f44ceb4771504342bb856d76c112d5a6,Magical School Boy and the Rock of Wi...,Fantasy|Growing up|Magic,2001,91,based on book,1414248543
43d02fb064514ff3bd30d1e3a7398357,Master of the Jewlery: The Company of...,Fantasy|Magic|Suspenseful,2001,19,adventure,1445286141
6aa0d26a483148998c250b9c80ddf550,Sun Conflicts: Part IV: A Novel Espair,Fantasy,1977,3,sci-fi,1439472256
6aa0d26a483148998c250b9c80ddf550,Sun Conflicts: Part IV: A Novel Espair,Fantasy,1977,87,sci-fi,1522676660
6aa0d26a483148998c250b9c80ddf550,Sun Conflicts: Part IV: A Novel Espair,Fantasy,1977,87,science fiction,1522676703
6aa0d26a483148998c250b9c80ddf550,Sun Conflicts: Part IV: A Novel Espair,Fantasy,1977,87,space,1522676664
eace16e59ce24eff90bf8924eb6a926c,The Outstanding Bulk,Fantasy|Superhero|Family,2008,87,bad science,1522676752


In [28]:
moviesWithTags
    .groupBy { movieId }
    .aggregate {
        title.first() into "title"
        tag.dropNulls().toSet() into "tags"
    }

movieId,title,tags
9b30aff7943f44579e92c261f3adc193,Women in Black,[Oscar (Best Supporting Actress)]
2a1ba1fc5caf492a80188e032995843e,Bumblebee Movie,[bah]
f44ceb4771504342bb856d76c112d5a6,Magical School Boy and the Rock of Wi...,"[fantasy, based on book]"
43d02fb064514ff3bd30d1e3a7398357,Master of the Jewlery: The Company of...,[adventure]
6aa0d26a483148998c250b9c80ddf550,Sun Conflicts: Part IV: A Novel Espair,"[sci-fi, science fiction, space]"
eace16e59ce24eff90bf8924eb6a926c,The Outstanding Bulk,[bad science]
ae916bc4844a4bb7b42b70d9573d05cd,In Automata,"[great dialogue, artificial intellige..."
c1f0a868aeb44c5ea8d154ec3ca295ac,Interplanetary,"[tense, post-apocalyptic, sci-fi, apo..."
9595b771f87f42a3b8dd07d91e7cb328,Woods Run,[classic]
aa9fc400e068443488b259ea0802a975,Anthropod-Dude,[quirky]


In [32]:
moviesWithTags
    .groupBy { movieId and title }.values(dropNA = true, distinct = true) { tag into "tags" }
    .sortByDesc { expr { "tags"<List<*>>().count() } }

movieId,title,tags
ae916bc4844a4bb7b42b70d9573d05cd,In Automata,"[great dialogue, artificial intellige..."
c1f0a868aeb44c5ea8d154ec3ca295ac,Interplanetary,"[tense, post-apocalyptic, sci-fi, apo..."
6aa0d26a483148998c250b9c80ddf550,Sun Conflicts: Part IV: A Novel Espair,"[sci-fi, science fiction, space]"
ee28d7e69103485c83e10b8055ef15fb,Metal Man 2,"[franchise, sci-fi, science fiction]"
f24327f2b05147b197ca34bf13ae3524,Krubit: Societal Teachings for Do Man...,"[dark comedy, so bad it's good, docum..."
f44ceb4771504342bb856d76c112d5a6,Magical School Boy and the Rock of Wi...,"[fantasy, based on book]"
8cf4d0c1bd7b41fab6af9d92c892141f,That Thing About an Iceberg,"[cliche, romantic]"
2bb29b3a245e434fa80542e711fd2cee,This is No Movie,"[musical, unpredictable]"
9b30aff7943f44579e92c261f3adc193,Women in Black,[Oscar (Best Supporting Actress)]
2a1ba1fc5caf492a80188e032995843e,Bumblebee Movie,[bah]


In [42]:
moviesWithTags
    .groupBy { movieId and title }.values(dropNA = true, distinct = true) { tag into "tags" }
    .sortByDesc { expr { "tags"<List<*>>().count() } }
    .take(10)

movieId,title,tags
ae916bc4844a4bb7b42b70d9573d05cd,In Automata,"[great dialogue, artificial intellige..."
c1f0a868aeb44c5ea8d154ec3ca295ac,Interplanetary,"[tense, post-apocalyptic, sci-fi, apo..."
6aa0d26a483148998c250b9c80ddf550,Sun Conflicts: Part IV: A Novel Espair,"[sci-fi, science fiction, space]"
ee28d7e69103485c83e10b8055ef15fb,Metal Man 2,"[franchise, sci-fi, science fiction]"
f24327f2b05147b197ca34bf13ae3524,Krubit: Societal Teachings for Do Man...,"[dark comedy, so bad it's good, docum..."
f44ceb4771504342bb856d76c112d5a6,Magical School Boy and the Rock of Wi...,"[fantasy, based on book]"
8cf4d0c1bd7b41fab6af9d92c892141f,That Thing About an Iceberg,"[cliche, romantic]"
2bb29b3a245e434fa80542e711fd2cee,This is No Movie,"[musical, unpredictable]"
9b30aff7943f44579e92c261f3adc193,Women in Black,[Oscar (Best Supporting Actress)]
2a1ba1fc5caf492a80188e032995843e,Bumblebee Movie,[bah]


In [43]:
val tagsPerMovie = moviesWithTags
    .groupBy { movieId and title }.values(dropNA = true, distinct = true) { tag into "tags" }
    .add("tagsCount") {
        "tags"<List<*>>().count()
    }
    .sortByDesc("tagsCount")
tagsPerMovie

movieId,title,tags,tagsCount
ae916bc4844a4bb7b42b70d9573d05cd,In Automata,"[great dialogue, artificial intellige...",6
c1f0a868aeb44c5ea8d154ec3ca295ac,Interplanetary,"[tense, post-apocalyptic, sci-fi, apo...",6
6aa0d26a483148998c250b9c80ddf550,Sun Conflicts: Part IV: A Novel Espair,"[sci-fi, science fiction, space]",3
ee28d7e69103485c83e10b8055ef15fb,Metal Man 2,"[franchise, sci-fi, science fiction]",3
f24327f2b05147b197ca34bf13ae3524,Krubit: Societal Teachings for Do Man...,"[dark comedy, so bad it's good, docum...",3
f44ceb4771504342bb856d76c112d5a6,Magical School Boy and the Rock of Wi...,"[fantasy, based on book]",2
8cf4d0c1bd7b41fab6af9d92c892141f,That Thing About an Iceberg,"[cliche, romantic]",2
2bb29b3a245e434fa80542e711fd2cee,This is No Movie,"[musical, unpredictable]",2
9b30aff7943f44579e92c261f3adc193,Women in Black,[Oscar (Best Supporting Actress)],1
2a1ba1fc5caf492a80188e032995843e,Bumblebee Movie,[bah],1


In [44]:
tagsPerMovie[0].tags

[great dialogue, artificial intelligence, philosophical, android(s)/cyborg(s), philosophical issues, thought-provoking]